# `gs_prompt_manager` — Full Tutorial

This notebook walks through every feature of the package end-to-end:

1. Install from PyPI and verify the version
2. `PromptBase` basics — variables, defaults, macros, metadata
3. Auto-extraction of variables
4. `PromptManager` — directory-based discovery
5. `PromptGroup` — auto-grouping by suffix
6. `@prompt_group` — explicit grouping
7. Solo prompts
8. Group querying patterns
9. Attribute-style access on `PromptManager`
10. End-to-end LLM-style usage
11. Cleanup

Every cell is intended to run top-to-bottom in a fresh kernel.

## 1. Install and verify

Pull the latest release from PyPI.

In [21]:
%pip install --quiet --upgrade gs_prompt_manager

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from importlib.metadata import version as _pkg_version
from packaging.version import Version

installed_version = _pkg_version("gs_prompt_manager")
print(f"Installed gs_prompt_manager version: {installed_version}")
assert Version(installed_version) >= Version("0.0.8"), (
    f"Expected 0.0.8 or newer; got {installed_version}. Re-run the pip install cell."
)

In [23]:
from gs_prompt_manager import PromptBase, PromptManager, PromptGroup, prompt_group

print("Exports OK:", PromptBase, PromptManager, PromptGroup, prompt_group)

Exports OK: <class 'gs_prompt_manager.prompt_base.PromptBase'> <class 'gs_prompt_manager.prompt_manager.PromptManager'> <class 'gs_prompt_manager.prompt_group.PromptGroup'> <function prompt_group at 0x000001723EEF32E0>


## 2. `PromptBase` basics

A prompt is a subclass of `PromptBase`. At minimum, override `set_prompt` to return the template string. The class name is used as the prompt name by default.

**Variables** (`{var}`) are user-supplied at call time (or via `set_variable_defaults`).  
**Macros** (`<<VAR>>`) are class-owned values defined in `set_macros` — not passed by callers.

In [ ]:
class Greeting(PromptBase):
    """A minimal prompt."""
    def set_prompt(self):
        return "Hello, {name}!"

    def set_variable_defaults(self):
        self.variable_defaults = {"name": "World"}

p = Greeting()
print("Default render:", p())                       # uses {name="World"}
print("Override:      ", p({"name": "Alice"}))     # caller supplies

### Required variables (no default)

A variable without a default must be supplied at call time, or `get_prompt` raises `ValueError`.

In [25]:
class Summarize(PromptBase):
    def set_prompt(self):
        return "Summarize: {text}"

    # No defaults -> {text} is required

p = Summarize()
try:
    p()  # missing required piece
except ValueError as e:
    print("Caught expected error:", e)

print("With value:", p({"text": "Lorem ipsum..."}))

ERROR:gs_prompt_manager.prompt_base:Prompt piece 'text' required in prompt input for Summarize; none given and no default.
NoneType: None


Caught expected error: Prompt piece 'text' required in prompt input for Summarize; none given and no default.
With value: Summarize: Lorem ipsum...


### Macros — `<<NAME>>`

Macros are class-owned values (e.g. timestamps, environment) substituted at render time. Use `<<NAME>>` syntax and override `set_macros`.

In [ ]:
import datetime

class LogLine(PromptBase):
    def set_prompt(self):
        return "[<<TIMESTAMP>>] {level}: {message}"

    def set_variable_defaults(self):
        self.variable_defaults = {"level": "INFO", "message": ""}

    def set_macros(self):
        self.macros = {
            "<<TIMESTAMP>>": datetime.datetime.now().isoformat(timespec="seconds"),
        }

log = LogLine()
print(log({"level": "ERROR", "message": "thing failed"}))

# Add a macro at runtime
log.add_macro("<<RUN_ID>>", "abc-123")
print("macros:", log.macros)

### Metadata

`get_metadata()` returns a serializable description: tags, version, author, examples, etc. helpful if the prompt is to be automatically selected by an LLM.

In [ ]:
import json

class Documented(PromptBase):
    def __init__(self):
        super().__init__(
            description="Summarize a code diff.",
            tags=["code", "diff", "summary"],
            author="guan@railtown.ai",
            version="1.2.0",
            example={"sample_variable": "diff text", "sample_response": "summary text"},
        )

    def set_prompt(self):
        return "Summarize:\n{diff}"

    def set_variable_defaults(self):
        self.variable_defaults = {"diff": ""}

meta = Documented().get_metadata()
print(json.dumps(
    {k: v for k, v in meta.items() if k in ("name", "description", "tags", "author", "version", "variable_defaults")},
    indent=2,
))

## 3. Auto-extraction of variables

If `variables` is not set explicitly, the base class extracts `{var}` names from the template automatically. Pair it with `set_variable_defaults_empty()` to give every extracted variable an empty default.

In [ ]:
class Compose(PromptBase):
    def set_prompt(self):
        return "Translate {text} from {source_lang} to {target_lang}."

    def set_variable_defaults(self):
        self.set_variable_defaults_empty()

c = Compose()
print("Auto-extracted variables:", c.variables)
print("Render:", c({"text": "hola", "source_lang": "es", "target_lang": "en"}))

## 4. `PromptManager` — directory-based discovery

`PromptManager` walks a directory and instantiates every `PromptBase` subclass it finds. For the notebook we'll write a few prompts into a temp dir on the fly.

In [ ]:
import tempfile, os, textwrap, shutil

workspace = tempfile.mkdtemp(prefix="gs_prompt_tutorial_")
print("Workspace:", workspace)

def write_prompt_file(name: str, body: str):
    path = os.path.join(workspace, name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(textwrap.dedent(body))
    return path

# A trio that demonstrates auto-grouping later
write_prompt_file("assistant.py", '''
    from gs_prompt_manager import PromptBase

    class AssistantSystem(PromptBase):
        """Sets persona for the LLM."""
        def set_prompt(self):
            return "You are a helpful assistant specialized in {domain}."
        def set_variable_defaults(self):
            self.variable_defaults = {"domain": "general knowledge"}

    class AssistantChat(PromptBase):
        """User-facing message body."""
        def set_prompt(self):
            return "{user_input}"
        def set_variable_defaults(self):
            self.variable_defaults = {"user_input": "Say hello."}
''')

# A standalone prompt (no recognized suffix -> solo group)
write_prompt_file("summary.py", '''
    from gs_prompt_manager import PromptBase

    class SummaryReport(PromptBase):
        def set_prompt(self):
            return "Report:\\n{body}"
        def set_variable_defaults(self):
            self.variable_defaults = {"body": ""}
''')

# An explicit @prompt_group example
write_prompt_file("greetings.py", '''
    from gs_prompt_manager import PromptBase, prompt_group

    @prompt_group("Greeting")
    class GreetingFormal(PromptBase):
        # key derived: strip "Greeting" prefix -> "formal"
        def set_prompt(self):
            return "Good day. How may I help you?"

    @prompt_group("Greeting", "casual")
    class HiThere(PromptBase):
        # explicit key -> "casual"
        def set_prompt(self):
            return "Hey! What\'s up?"
''')

print("Files written:", os.listdir(workspace))

In [30]:
manager = PromptManager(prompt_paths=workspace)

print("Loaded prompts:")
for name in manager.get_prompt_names():
    print("  -", name)

print("\nResolved groups:")
for gname, group in manager.get_prompt_groups().items():
    print(f"  - {gname}: keys={group.get_prompt_names()}")

Loaded prompts:
  - AssistantChat
  - AssistantSystem
  - GreetingFormal
  - HiThere
  - SummaryReport

Resolved groups:
  - Assistant: keys=['chat', 'system']
  - Greeting: keys=['formal', 'casual']
  - SummaryReport: keys=['default']


In [31]:
# Get a single prompt by class name
asst_chat = manager.get_prompt("AssistantChat")
print(asst_chat({"user_input": "What's a decorator?"}))

What's a decorator?


## 5. `PromptGroup` — auto-grouping by suffix

When a class name ends in one of `System`, `Chat`, `Pre`, `Post`, `Message`, `Prompt` (case-insensitive, with or without a leading underscore), `PromptManager` bundles it into a group named after the stem. The key in the group is the lowercased suffix.

| Class | Group | Key |
|-------|-------|-----|
| `AssistantSystem` | `Assistant` | `system` |
| `AssistantChat`   | `Assistant` | `chat`   |
| `Worker_pre`      | `Worker`    | `pre`    |
| `ExampleMessage`  | `Example`   | `message`|

In [32]:
assistant = manager.get_prompt_group("Assistant")
print("Group:", assistant)
print("Members:", assistant.get_prompt_names())
print("---")
print("system :", assistant.system({"domain": "Python"}))
print("chat   :", assistant.chat({"user_input": "Explain context managers."}))

Group: Say hello.
Members: ['chat', 'system']
---
system : You are a helpful assistant specialized in Python.
chat   : Explain context managers.


## 6. `@prompt_group` — explicit grouping

Use the decorator when the suffix convention doesn't apply, or when you want a non-obvious group name or key. Key derivation when no explicit key is given: strip the group-name prefix (case-insensitive), remove underscores, lowercase. If nothing remains, the key falls back to `"default"`.

In [33]:
greeting = manager.get_prompt_group("Greeting")
print("Members:", greeting.get_prompt_names())
print("formal :", greeting.formal())
print("casual :", greeting.casual())

Members: ['formal', 'casual']
formal : Good day. How may I help you?
casual : Hey! What's up?


## 7. Solo prompts

A prompt with no decorator and no recognized suffix becomes its own one-member group with key `"default"`. Every loaded prompt is reachable via `get_prompt_group`, even ones you didn't deliberately group.

In [34]:
solo = manager.get_prompt_group("SummaryReport")
print("Members:", solo.get_prompt_names())
print(solo.default({"body": "Q3 revenue up 12%."}))

Members: ['default']
Report:
Q3 revenue up 12%.


## 8. Group querying patterns

Three equivalent ways to reach a member, plus container-style checks.

In [35]:
g = manager.get_prompt_group("Assistant")


# attribute, item, and method access all return the same PromptBase
print(f"Method 1 result: {g.system}")
print(f"Method 2 result: {g['system']}")
print(f"Method 3 result: {g.get_prompt('system')}") # as if prompt_group is an manager too!
assert g.system is g["system"] is g.get_prompt("system")

# container protocol
print("len:", len(g))
print("'chat' in group:", "chat" in g)
print("iter:", list(iter(g)))

# str(group) renders one member. Priority: 'default' -> 'chat' -> first available
print("\nstr(g):\n" + str(g))

Method 1 result: You are a helpful assistant specialized in general knowledge.
Method 2 result: You are a helpful assistant specialized in general knowledge.
Method 3 result: You are a helpful assistant specialized in general knowledge.
len: 2
'chat' in group: True
iter: ['chat', 'system']

str(g):
Say hello.


## 9. Attribute-style access on `PromptManager`

`PromptManager` supports attribute-style access so you can skip the explicit `get_prompt_group` / `get_prompt` call. **Groups take priority** — since every prompt lands in at least one group, most attribute accesses return a `PromptGroup`.

`dir(manager)` includes all group and prompt names, so tab-completion works in notebooks.

In [ ]:
# Attribute access returns the PromptGroup (groups take priority)
asst_group = manager.Assistant
print(type(asst_group))                        # <class 'PromptGroup'>
assert asst_group is manager.get_prompt_group("Assistant")

# Chain straight to a variant
print(manager.Assistant.system({"domain": "Python"}))
print(manager.Greeting.casual())

# Solo prompt is also accessible via its group name
solo = manager.SummaryReport
print(type(solo))                              # PromptGroup (solo group, key "default")
print(solo.default({"body": "Q3 up 12%."}))

# dir() includes group and prompt names
names_in_dir = [n for n in dir(manager) if not n.startswith("_")]
print("group/prompt names visible in dir:", [n for n in names_in_dir if n[0].isupper()])

# AttributeError for unknown names
try:
    _ = manager.DoesNotExist
except AttributeError as e:
    print("Expected AttributeError:", e)

## 10. End-to-end LLM-style usage

Putting it together: this is what calling code typically looks like when feeding a chat-completion API. Replace the `fake_llm` stub with a real client call to OpenAI, Anthropic, etc.

In [36]:
def fake_llm(messages):
    """Stand-in for an LLM API call. Returns a deterministic echo for the tutorial."""
    sys = next(m['content'] for m in messages if m['role'] == 'system')
    usr = next(m['content'] for m in messages if m['role'] == 'user')
    return f"[ASSISTANT replying under: {sys!r}] {usr!r}"

asst = manager.get_prompt_group("Assistant")

messages = [
    {"role": "system", "content": asst.system({"domain": "Python"})},
    {"role": "user",   "content": asst.chat({"user_input": "What does @staticmethod do?"})},
]

print(fake_llm(messages))

[ASSISTANT replying under: 'You are a helpful assistant specialized in Python.'] 'What does @staticmethod do?'


### Inspecting prompts loaded into a manager

Useful when debugging — list every group and its members:

In [37]:
for gname, group in manager.get_prompt_groups().items():
    print(f"[{gname}]  keys: {group.get_prompt_names()}")
    for key, prompt in group.get_prompts().items():
        print(f"   .{key:8s}  ->  {type(prompt).__name__}: {prompt.prompt!r}")

[Assistant]  keys: ['chat', 'system']
   .chat      ->  AssistantChat: '{user_input}'
   .system    ->  AssistantSystem: 'You are a helpful assistant specialized in {domain}.'
[Greeting]  keys: ['formal', 'casual']
   .formal    ->  GreetingFormal: 'Good day. How may I help you?'
   .casual    ->  HiThere: "Hey! What's up?"
[SummaryReport]  keys: ['default']
   .default   ->  SummaryReport: 'Report:\n{body}'


## 11. Cleanup

In [38]:
shutil.rmtree(workspace)
print("Removed workspace:", workspace)

Removed workspace: C:\Users\guanz\AppData\Local\Temp\gs_prompt_tutorial_1w7e28ab


---

**Recap**

- `PromptBase`: subclass + `set_prompt` is enough. Variables (`{var}`) come from callers or `set_variable_defaults`; macros (`<<VAR>>`) belong to the class via `set_macros`.
- `PromptManager`: discovers `PromptBase` subclasses in a directory and instantiates them.
- `PromptGroup`: bundles related variants. Resolution priority is `@prompt_group` → suffix auto-detect → solo (key `"default"`).
- Access patterns on a group: `group.system`, `group["system"]`, `group.get_prompt("system")` are equivalent.
- Access patterns on a manager: `manager.Assistant` (attribute) == `manager.get_prompt_group("Assistant")` (explicit). Groups take priority; `dir(manager)` lists all available names.

See [docs/user-guide.md](user-guide.md) and [docs/examples.md](examples.md) for deeper coverage.